# Lab 1: LangChain Introduction

## The LangChain ecosystem

The ecosystem is organized into three layers, each built on top of the previous one:

1. **LangChain** provides the high-level building blocks. It offers a unified interface for chat models from multiple providers (OpenAI, Anthropic, Google, local models, ...), together with abstractions for prompts, messages, tools, structured output, retrievers and document loaders. It also provides `create_agent`, a predefined agent loop (model → tool calls → model → ...) that can be configured with minimal code.

2. **LangGraph** is the low-level orchestration runtime. An application is modeled as a **graph** consisting of a shared **state**, **nodes** (functions or LLM calls) that update this state, and **edges** (fixed or conditional) that determine the next step of execution. Because graphs may contain cycles, loops, retries and multi-agent workflows can be expressed directly. LangGraph additionally supplies the capabilities required by production systems: **persistence/checkpointing**, **memory**, **streaming**, **human-in-the-loop** interrupts and **durable execution**. The agents provided by LangChain are themselves implemented on top of LangGraph.

3. **Deep Agents** (`deepagents`) is an "agent harness" built on LangGraph and intended for long-running, multi-step tasks. By default, it includes a **planning tool** (a to-do list), a **virtual file system** for managing context, support for delegating focused subtasks to **sub-agents**, and detailed system prompts. This architecture follows the pattern adopted by systems such as Claude Code and Deep Research.

```
Deep Agents   →  opinionated harness for complex, long-horizon agents
     ↑
LangGraph     →  stateful graphs: control flow, persistence, human-in-the-loop
     ↑
LangChain     →  model / tool / prompt abstractions, quick-start agents
```

### Why LangGraph matters beyond the framework

Software libraries in this field evolve rapidly: APIs are renamed, packages are restructured, and new frameworks emerge frequently. **The concepts underlying LangGraph, however, remain relevant well beyond the lifetime of any particular library:**

- An agent can be formalized as a **state machine**: an explicit state, a set of operations that transform it, and transitions that determine the subsequent step.
- **Control flow should be explicit.** The designer determines where the LLM is permitted to make decisions and where the logic remains deterministic.
- **Loops, branching, retries and stopping conditions** are the mechanisms that make agents reliable, in contrast to single-prompt approaches.
- **Checkpointing, memory and human approval** are the features that distinguish a prototype from a trustworthy system.

A conceptual understanding of LLM applications as graphs of state transitions enables the analysis, design and debugging of agents in *any* framework (LangGraph, other agent SDKs, or custom implementations). This lab begins with the fundamentals of LangChain.


## Anatomy of an agent: `agent = model + harness`

![Agent = model + harness: the model surrounded by skills, tools, context, subagents, system prompt and memory](images/agent_model_harness.svg)

*Figure 1. The components of an agent: the model at the center, surrounded by the elements of the harness. Source: LangChain, "Agents", LangChain Documentation, https://docs.langchain.com/oss/python/langchain/agents (accessed 8 October 2026).*

An agent can be described as the combination of a **model** and a **harness**. The model is the large language model that performs the reasoning and selects the next action. The harness is the surrounding software that supplies the model with instructions, capabilities and information, and that executes the actions the model selects. The model determines the *quality of each decision*; the harness determines *what the model can perceive and act upon*. The core components of the harness are the following:

- **Model.** The LLM at the center of the agent. At each step, it receives the current input and produces either a final answer or a request to invoke one or more tools. LangChain provides a unified interface that allows models from different providers to be interchanged.

- **System prompt.** The persistent instructions that define the agent's role, objectives, constraints and expected behavior. It is included in every model call and constitutes the primary mechanism for steering the agent.

- **Tools.** Functions with a defined name, description and input schema that the model can request to invoke, for example web search, database queries, code execution or calls to external APIs. Tools allow the agent to obtain information and to act on its environment beyond text generation.

- **Context.** The complete information available to the model at a given step: the system prompt, the conversation history, tool results, retrieved documents and intermediate state. Since the context window is finite, *context engineering*, i.e. selecting, summarizing and organizing this information, is a central concern in agent design.

- **Memory.** The mechanism through which information persists beyond a single model call. *Short-term memory* preserves the state of the current conversation or task (in LangGraph, through checkpointing). *Long-term memory* stores facts, preferences or past experiences across sessions so that they can be retrieved later.

- **Skills.** Packaged units of specialized knowledge, such as instructions, reference material and scripts, that the agent loads only when a task requires them. This keeps the context compact while giving the agent access to domain-specific procedures on demand.

- **Subagents.** Separate agents, each with its own prompt, tools and isolated context, to which the main agent delegates focused subtasks. Delegation keeps the main agent's context clean and allows complex problems to be decomposed, as in the Deep Agents architecture described above.

From this perspective, the frameworks discussed above differ mainly in the harness they provide. LangChain supplies the model and tool abstractions, LangGraph supplies the control flow, state and memory, and Deep Agents supplies a preconfigured harness that combines planning, context management, skills and subagents.


## Working with models

The **model** is the reasoning component of every LangChain application. LangChain exposes chat models from different providers through a common interface, so that the rest of the application remains unchanged when the provider is replaced. Two approaches are available for creating a model:

1. **`init_chat_model`**, a provider-independent factory function from `langchain.chat_models`. The model can be specified either as a single string in the format `"{model_provider}:{model}"` (for example, `"openai:gpt-5.5"`) or through the separate `model` and `model_provider` parameters.
2. **A provider-specific class**, such as `ChatGoogleGenerativeAI`, `ChatOpenAI` or `ChatGroq`, imported from the corresponding integration package (`langchain-google-genai`, `langchain-openai`, `langchain-groq`, ...). This approach exposes all provider-specific options.

In both cases, the integration package of the selected provider must be installed.

### Standard parameters

| Parameter | Description |
|---|---|
| `model` | The name or identifier of the model (e.g. `"gemini-3.8-flash"`). |
| `model_provider` | The provider of the model (e.g. `"google_genai"`, `"openai"`, `"groq"`, `"mistralai"`). It may be omitted when it can be inferred from the model name or when the `provider:model` format is used. |
| `api_key` | The authentication key for the provider. It is usually read from an environment variable (e.g. `GOOGLE_API_KEY`) and therefore rarely passed explicitly. |
| `temperature` | Controls the randomness of the output. Low values (close to 0) yield more deterministic answers; higher values yield more varied ones. |
| `max_tokens` | The maximum number of tokens in the generated response. |
| `timeout` | The maximum time, in seconds, to wait for a response before the request is cancelled. |
| `max_retries` | The maximum number of attempts for a failed request (e.g. due to network errors or rate limits). |
| `base_url` | A custom API endpoint, used for OpenAI-compatible providers such as OpenRouter or for locally hosted models. |

Additional, provider-specific parameters may be passed as keyword arguments.

### Invoking a model

- **`invoke()`** sends a single input (a string or a list of messages) and returns the complete response as an `AIMessage`; the generated text is available in `response.content`.
- **`stream()`** returns an iterator that yields the response in chunks as they are generated, which is useful for long outputs and interactive interfaces.
- **`batch()`** processes several independent inputs in parallel.

In addition, `init_chat_model` can create *configurable* models (through the `configurable_fields` parameter), whose model or provider is selected at invocation time rather than at creation time.

*Source: LangChain, "Models", LangChain Documentation, https://docs.langchain.com/oss/python/langchain/models (accessed 8 October 2026).*


## Obtaining an API key

Access to a hosted model requires an API key from the corresponding provider. The following providers are suitable for this laboratory:

| Provider | Access | API key page | `model_provider` / integration | Environment variable |
|---|---|---|---|---|
| **Google AI Studio** (Gemini) | Free tier | https://aistudio.google.com/api-keys | `google_genai` / `langchain-google-genai` | `GOOGLE_API_KEY` |
| **Groq** | Free tier | https://console.groq.com/keys | `groq` / `langchain-groq` | `GROQ_API_KEY` |
| **Mistral AI** | Free tier (*Experiment* plan) | https://console.mistral.ai/api-keys | `mistralai` / `langchain-mistralai` | `MISTRAL_API_KEY` |
| **OpenRouter** | Free models (identified by the `:free` suffix) | https://openrouter.ai/keys | OpenAI-compatible API, `base_url="https://openrouter.ai/api/v1"` | `OPENROUTER_API_KEY` |
| **OpenAI** | Paid (pay-as-you-go) | https://platform.openai.com/api-keys | `openai` / `langchain-openai` | `OPENAI_API_KEY` |

Remarks:

- Free tiers are subject to **rate limits** (requests per minute and per day), and the available models and limits change frequently. The current conditions should be consulted on each provider's website.
- Under some free tiers, the submitted data **may be used to improve the provider's models**. Confidential or personal data should therefore not be sent through free-tier keys.
- API keys must be stored in a local `.env` file (e.g. `GOOGLE_API_KEY=...`) and **must never be committed** to a version control repository. The `.env` file should be listed in `.gitignore`.


In [ ]:
!pip install -q -r requirements.txt

## A first model call

The following code cell performs the same request with the two approaches described above:

1. **Loading the API key.** `load_dotenv()` reads the variables defined in the local `.env` file into the process environment. The code then reports whether `GOOGLE_API_KEY` has been found; if it is missing, `getpass.getpass()` prompts for the key interactively without displaying it on screen.
2. **Defining the prompt.** The prompt states the task (explaining relational databases and SQL), a length constraint (at most three sentences) and the intended audience (a computer science student). Such explicit constraints make the output shorter, more focused and easier to compare across models than an open-ended greeting.
3. **Provider-independent initialization.** `init_chat_model(model="gemini-3.8-flash", model_provider="google_genai", temperature=0.7)` creates a Gemini chat model through the generic factory function. A temperature of `0.7` allows a moderate degree of variation in the answers.
4. **First invocation.** `model.invoke(prompt)` sends the prompt as a single user message and returns an `AIMessage`; its text is printed through `response.content`.
5. **Provider-specific initialization.** The same model is then created directly with the `ChatGoogleGenerativeAI` class from `langchain-google-genai` and invoked with the same prompt.

Both objects expose the same interface (`invoke`, `stream`, `batch`). Consequently, replacing Gemini with another provider from the table above requires only a different `model` / `model_provider` pair and the corresponding API key. The remainder of the code remains unchanged.


In [ ]:
import os
import getpass
from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv
load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    print("API key for GOOGLE AI API is not set. Please check your .env file.")
else:
    print("API key for GOOGLE AI API loaded successfully.")

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

prompt = (
    "In at most three sentences, explain what a relational database is "
    "and why SQL is used to query it. Address the explanation to a "
    "computer science student."
)

model = init_chat_model(
    model="gemini-3.8-flash",
    model_provider="google_genai",
    temperature=0.7
)

response = model.invoke(prompt)
print(response.content)

model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash"
)

response = model.invoke(prompt)
print(response.content)


## Messages: `SystemMessage`, `HumanMessage` and `AIMessage`

A string passed to `invoke()` is converted internally into a single user message. In practice, the input to a chat model is a **list of messages**, each associated with a role:

| Class | Role | Purpose |
|---|---|---|
| `SystemMessage` | system | Defines the model's role, the task, the rules to follow and the available context (e.g. a database schema). It is usually placed first. |
| `HumanMessage` | user | Contains the input of the user, such as a question or an instruction. |
| `AIMessage` | assistant | Contains a response of the model. It is returned by `invoke()`, and it may also be added to the input to represent earlier turns of the conversation or worked examples. |

The following example applies these message types to a **text-to-SQL** task: the model receives a natural-language question about Formula 1 and produces an SQLite query, which is then executed on the database `data/formula_1/formula_1.sqlite` (from the BIRD benchmark). The database contains tables such as `drivers`, `constructors`, `races`, `circuits` and `results`; textual descriptions of all columns are available in `data/formula_1/database_description/`.

The first step establishes a connection to the database through a **SQLAlchemy `Engine`** and extracts the schema (tables, columns, types and foreign keys) as text, so that it can be supplied to the model in the system message.


In [ ]:
from pathlib import Path

from sqlalchemy import create_engine, inspect, text
from sqlalchemy.engine import Engine

DB_PATH = Path("data/formula_1/formula_1.sqlite")
assert DB_PATH.exists(), f"Database not found: {DB_PATH.resolve()}"

engine: Engine = create_engine(f"sqlite:///{DB_PATH.as_posix()}")


def get_schema(engine: Engine) -> str:
    """Return a textual description of all tables, columns and foreign keys."""
    inspector = inspect(engine)
    lines = []
    for table in inspector.get_table_names():
        columns = ", ".join(
            f"{col['name']} {col['type']}" for col in inspector.get_columns(table)
        )
        lines.append(f"{table}({columns})")
        for fk in inspector.get_foreign_keys(table):
            lines.append(
                f"  FOREIGN KEY ({', '.join(fk['constrained_columns'])}) "
                f"REFERENCES {fk['referred_table']}({', '.join(fk['referred_columns'])})"
            )
    return "\n".join(lines)


def run_sql(engine: Engine, sql: str, max_rows: int = 10) -> None:
    """Execute a read-only query and print the column names and the first rows."""
    if not sql.lstrip().upper().startswith(("SELECT", "WITH")):
        raise ValueError("Only SELECT queries are allowed.")
    with engine.connect() as conn:
        result = conn.execute(text(sql))
        print(list(result.keys()))
        for row in result.fetchmany(max_rows):
            print(tuple(row))


schema = get_schema(engine)
print(schema)


### Generating SQL from a message list

The next cell constructs the input to the model as a list of messages:

1. A **`SystemMessage`** defines the role of the model (an SQLite assistant), the rules it must follow (read-only queries, schema-conformant names, output limited to the SQL statement) and the database schema obtained above.
2. A **`HumanMessage` / `AIMessage` pair** provides a worked example (*few-shot prompting*): a question together with the expected answer. The `AIMessage` is written by the developer rather than generated by the model; it demonstrates the required output format.
3. A final **`HumanMessage`** contains the actual question.

A separate model instance with `temperature=0` is used, since SQL generation requires deterministic and precise output rather than varied text. The returned `AIMessage` is then cleaned of possible Markdown code fences and executed on the database through `run_sql`.


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

sql_model = init_chat_model(
    model="gemini-3.8-flash",
    model_provider="google_genai",
    temperature=0
)


def extract_sql(response: AIMessage) -> str:
    """Return the SQL text of a response, without Markdown code fences."""
    sql = response.text.strip()
    if sql.startswith("```"):
        sql = sql.split("\n", 1)[1].rsplit("```", 1)[0]
    return sql.strip()


system_message = SystemMessage(content=f"""You are an expert SQLite assistant for a Formula 1 database.
Translate the user's question into a single SQLite query.

Rules:
- Use only the tables and columns listed in the schema below.
- Generate read-only queries (SELECT) only.
- Return only the SQL query, without explanations or Markdown formatting.

Database schema:
{schema}""")

messages = [
    system_message,
    # Worked example (few-shot): a question and the expected answer
    HumanMessage(content="How many races were held in 2009?"),
    AIMessage(content="SELECT COUNT(*) AS race_count FROM races WHERE year = 2009;"),
    # Actual question
    HumanMessage(content="Which driver won the most races in the 2011 season? "
                         "Return the driver's full name and the number of wins."),
]

response = sql_model.invoke(messages)
sql = extract_sql(response)
print(sql, end="\n\n")
run_sql(engine, sql)


### Continuing the conversation

Chat models are **stateless**: each call receives only the messages passed to it. A multi-turn conversation is therefore represented by appending the model's response (an `AIMessage`) and the next `HumanMessage` to the existing list. In the following cell, the follow-up question refers to "the same question" without restating it; the model can resolve this reference only because the previous question and its generated query are part of the input.

This explicit management of the message history is precisely the kind of *state* that LangGraph later handles automatically through its state and memory mechanisms.


In [ ]:
messages.append(response)  # the AIMessage returned by the model
messages.append(HumanMessage(content="Answer the same question for constructors (teams) "
                                     "instead of drivers. Return the constructor name and the number of wins."))

response = sql_model.invoke(messages)
sql = extract_sql(response)
print(sql, end="\n\n")
run_sql(engine, sql)

print(f"\nThe conversation now contains {len(messages)} input messages:")
for message in messages:
    print(f"- {type(message).__name__}")


## Exercises

The following exercises extend the text-to-SQL example above. They rely on the code presented in this notebook and on the helper functions defined in the next cell:

| Function | Purpose |
|---|---|
| `fetch_rows(engine, sql)` | Executes a read-only query and **returns** the rows as a list of tuples (`run_sql` only prints them). |
| `same_result(engine, predicted_sql, gold_sql)` | Returns `True` if both queries produce the same set of rows. This is the *execution accuracy* criterion of the BIRD benchmark. |
| `load_bird_questions(db_id)` | Loads the BIRD questions for one database from `data/dev.json`. Each item contains a `question`, an `evidence` (a hint), the reference `SQL` and a `difficulty` level. |
| `load_column_descriptions(db_dir)` | Returns the column descriptions from the `database_description/` folder as text, so that they can be added to the system message. |

**Note.** Free API tiers impose rate limits. When many questions are processed in a loop, a short pause between calls (e.g. `time.sleep(4)`) prevents request errors.


In [ ]:
import csv
import json


def fetch_rows(engine: Engine, sql: str) -> list[tuple]:
    """Execute a read-only query and return all rows as a list of tuples."""
    if not sql.lstrip().upper().startswith(("SELECT", "WITH")):
        raise ValueError("Only SELECT queries are allowed.")
    with engine.connect() as conn:
        return [tuple(row) for row in conn.execute(text(sql))]


def same_result(engine: Engine, predicted_sql: str, gold_sql: str) -> bool:
    """Compare two queries by their result sets (BIRD execution accuracy)."""
    try:
        return set(fetch_rows(engine, predicted_sql)) == set(fetch_rows(engine, gold_sql))
    except Exception:
        return False


def load_bird_questions(db_id: str, path: str = "data/dev.json") -> list[dict]:
    """Load the BIRD questions that belong to one database."""
    with open(path, encoding="utf-8") as f:
        return [item for item in json.load(f) if item["db_id"] == db_id]


def load_column_descriptions(db_dir: Path) -> str:
    """Return the column descriptions of all tables as text, one column per line."""
    lines = []
    for csv_path in sorted((db_dir / "database_description").glob("*.csv")):
        with open(csv_path, encoding="utf-8-sig", errors="replace") as f:
            for row in csv.DictReader(f):
                row = {key.strip(): " ".join((value or "").split()) for key, value in row.items() if key}
                line = f"{csv_path.stem}.{row['original_column_name']}: {row['column_description']}"
                if row.get("value_description"):
                    line += f" ({row['value_description']})"
                lines.append(line)
    return "\n".join(lines)


f1_questions = load_bird_questions("formula_1")
print(f"{len(f1_questions)} Formula 1 questions loaded. Example:")
print(json.dumps(f1_questions[0], indent=2))


In [ ]:
# fetch_rows: execute the reference query of the first BIRD question
item = f1_questions[0]
rows = fetch_rows(engine, item["SQL"])
print(f"Question: {item['question']}")
print(f"Reference SQL: {item['SQL']}")
print(f"Rows returned: {len(rows)}, first rows: {rows[:5]}\n")

# fetch_rows: non-SELECT statements are rejected
try:
    fetch_rows(engine, "DELETE FROM races")
except ValueError as error:
    print(f"Rejected as expected: {error}\n")

# same_result: different SQL with the same result, a different result, and an invalid query
reference = "SELECT COUNT(*) FROM races WHERE year = 2009"
print("Equivalent query:", same_result(engine, "SELECT COUNT(raceId) FROM races WHERE year BETWEEN 2009 AND 2009", reference))
print("Different result:", same_result(engine, "SELECT COUNT(*) FROM races WHERE year = 2010", reference))
print("Invalid query:   ", same_result(engine, "SELECT COUNT(*) FROM race_list", reference), "\n")

# load_column_descriptions: column descriptions for the Formula 1 database
descriptions = load_column_descriptions(DB_PATH.parent)
print(f"{len(descriptions.splitlines())} column descriptions loaded. First lines:")
print("\n".join(descriptions.splitlines()[:5]))


### Exercise 1: A different database

*Objective: to observe that the prompt must be adapted to the domain of the data.*

1. Connect the engine to `data/student_club/student_club.sqlite`.
2. Adapt the system message and the few-shot example to the new domain.
3. Formulate three questions of your own.
4. Verify each generated query manually: does it answer the question correctly?

### Exercise 2: Prompt variations

*Objective: to measure the effect of the prompt, changing one factor at a time.*

Select five questions from `f1_questions`. For each configuration below, count how many generated queries return the same result as the reference query (use `same_result`):

1. **Few-shot examples:** zero, one and three `HumanMessage` / `AIMessage` pairs.
2. **Context:** the schema only vs. the schema plus the column descriptions (`load_column_descriptions`).
3. **Temperature:** `0` vs. `0.7`. Run each question three times and note whether the query changes.

Present the results in a table (configuration → number of correct answers).

### Exercise 3: Evaluation against reference queries

*Objective: to evaluate the model objectively, using execution accuracy.*

1. Select the first 20 questions from `f1_questions`.
2. Generate a query for each question.
3. Compare it with the reference query (`item["SQL"]`) using `same_result`.
4. Compute the execution accuracy: the number of correct queries divided by 20.
5. Repeat the evaluation with the hint (`item["evidence"]`) added to the `HumanMessage`. Compare the two accuracies.

### Exercise 4: Error analysis

*Objective: to understand why the model fails, not only how often.*

1. Collect the questions that failed in Exercise 3.
2. Compare each generated query with the reference query.
3. Classify each error, for example: wrong table or column, missing `JOIN`, wrong aggregation, wrong filter, or a misinterpreted question.
4. Select the most frequent category and modify the system message to prevent it.
5. Run the evaluation again and report whether the accuracy improved.

### Exercise 5: Self-correction

*Objective: to use the message history as feedback for the model.*

When a generated query fails with an error (call `fetch_rows` inside a `try` / `except` block to obtain the error text):

1. Append the query as an `AIMessage`.
2. Append the error text as a `HumanMessage` and ask for a corrected query.
3. Repeat at most three times.

Apply this procedure to the failed questions from Exercise 3. How many are corrected? Explain why queries that execute successfully but return a wrong result are not corrected.

### Exercise 6: Answers in natural language

*Objective: to chain two model calls and to keep the answer grounded in the data.*

1. Execute a generated query and obtain its rows with `fetch_rows`.
2. Make a second model call. The `SystemMessage` requires an answer based **only** on the given rows. The `HumanMessage` contains the question, the SQL query and the rows.
3. Print a one-sentence answer.
4. Compare the answer with the rows. Does it contain any information that is not present in the data?


## References

**Official documentation**
- LangChain. *LangChain Overview*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/overview
- LangChain. *Agents*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/agents (source of Figure 1; accessed 8 October 2026)
- LangChain. *Models*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/models (accessed 8 October 2026)
- LangChain. *LangGraph Overview*. LangGraph Documentation. https://docs.langchain.com/oss/python/langgraph/overview
- LangChain. *Deep Agents Overview*. Deep Agents Documentation. https://docs.langchain.com/oss/python/deepagents/overview

**Source code repositories**
- LangChain: https://github.com/langchain-ai/langchain
- LangGraph: https://github.com/langchain-ai/langgraph
- Deep Agents: https://github.com/langchain-ai/deepagents

**Courses and articles**
- LangChain Academy. *Introduction to LangGraph*. Online course. https://academy.langchain.com/courses/intro-to-langgraph
- Chase, H. (2025). *Deep Agents*. LangChain Blog. https://blog.langchain.com/deep-agents/
- Schluntz, E., & Zhang, B. (2024). *Building Effective Agents*. Anthropic Engineering Blog. https://www.anthropic.com/engineering/building-effective-agents

**Research papers**
- Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models*. International Conference on Learning Representations (ICLR). https://arxiv.org/abs/2210.03629
- Li, J., Hui, B., Qu, G., et al. (2023). *Can LLM Already Serve as A Database Interface? A BIg Bench for Large-Scale Database Grounded Text-to-SQLs* (BIRD). Advances in Neural Information Processing Systems (NeurIPS), Datasets and Benchmarks Track. https://arxiv.org/abs/2305.03111 (source of the Formula 1 database)


> **Note on the preparation of this document.** Parts of this laboratory guide were drafted with the assistance of Claude (Anthropic), an AI assistant, used through Claude Code.